# 03 — Modeling and Evaluation

Walk-forward validation of Naive Baseline, ARIMA/SARIMA, and Prophet across all three forecast horizons (1, 5, 21 trading days). Phase 7 (backtest) will complete this notebook.

## Phase 5 — ARIMA / SARIMA

**Context from earlier phases:**
- Phase 3: VIX levels are stationary → d = 0
- Phase 2: PACF cuts off after lag 1–2 → low AR order expected; ACF shows very slow decay (AR coefficient near 1)

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

from src.vix_forecasting.data.preprocessing import load_raw, build_series
from src.vix_forecasting.models.arima import select_order, ARIMAForecaster
from src.vix_forecasting.models.prophet_model import ProphetForecaster

vix = build_series(load_raw(Path('../data/raw/vix_raw.csv')))
sel = vix.iloc[-2000:]   # 2018-07-05 to present — order-selection window
print(f'Full series : {len(vix):,} obs  ({vix.index[0].date()} to {vix.index[-1].date()})')
print(f'Selection   : {len(sel):,} obs  ({sel.index[0].date()} to {sel.index[-1].date()})')

## Order selection — non-seasonal

Grid search over ARIMA(p,0,q) with p ∈ {1,2,3}, q ∈ {0,1,2} using AIC. d=0 is fixed (Phase 3 result).

In [ ]:
df_ns = select_order(sel, p_values=(1,2,3), q_values=(0,1,2), d=0, seasonal=False)
df_ns[['order','aic','bic','converged']]

**Findings:**

- The raw AIC winner is ARIMA(3,0,2), but it **did not converge** — its parameter estimates are unreliable.
- The best **converged** model is **ARIMA(1,0,2)** (AIC=8483). Consistent with the PACF: a dominant AR(1) plus short MA corrections.
- ARIMA(2,0,0) and ARIMA(3,0,0) also converge but score 9–17 AIC points worse — the MA terms carry real signal.

## Seasonality test — ARIMA SMA(5) at weekly frequency

In [ ]:
df_s = select_order(sel, p_values=(1,), q_values=(2,), d=0,
                    seasonal=True, m=5, P_values=(0,1), Q_values=(0,1))
df_s[['order','seasonal_order','aic','bic','converged']]

**Seasonality verdict — negative result:**

SMA(1) at lag 5 improves the 8-year selection-window AIC by ~17 points. However, when fitted on the full 36-year history the SMA(5) coefficient is **0.0005 with p=0.923** — effectively zero. Spurious artefact of the selection window, not a structural feature of VIX.

## Final ARIMA model: ARIMA(1, 0, 2)

In [ ]:
arima = ARIMAForecaster()   # default = (1,0,2)
arima.fit(vix)
print(arima._result.summary().tables[1])
print()
print(f'Implied long-run mean: {arima._result.params["intercept"] / (1 - arima._result.params["ar.L1"]):.2f}')
print(f'Historical mean      : {vix.mean():.2f}')

**Coefficient interpretation:**

- **AR(1) ≈ 0.984**: Very high persistence — today's level dominates tomorrow's forecast. Mean reversion is slow.
- **MA(1), MA(2)**: Both significant, correcting for short-term autocorrelation in residuals after the AR step.
- **Implied long-run mean ≈ 19.3**, close to the historical mean of 19.5 — the model is anchored correctly.

In [ ]:
print(f'Current VIX ({vix.index[-1].date()}): {vix.iloc[-1]:.2f}')
for h in (1, 5, 21):
    p = arima.predict(h)
    print(f'  horizon={h:2d}d  [{p[0]:.3f} ... {p[-1]:.3f}]')

---

## Phase 6 — Prophet

Prophet serves two purposes here:
1. **Seasonality cross-check** — does Prophet's independent decomposition confirm the ARIMA finding that weekly seasonality is negligible?
2. **Competitor model** — will be compared against ARIMA and the Naive baseline in the Phase 7 walk-forward backtest.

Configuration: additive mode (appropriate for a stationary series), weekly and yearly seasonality enabled, no daily seasonality (no intraday data).

---

## Phase 7 — Walk-Forward Backtest

**Setup:**
- Test window: last 756 obs (~3 years, 2023-06 to 2026-06)
- Step: 21 obs (monthly refit) · Folds: 36
- Horizons: 1, 5, 21 trading days
- Metrics: RMSE, MAE, MAPE, Directional Accuracy

In [ ]:
import pandas as pd

# Results pre-computed by reports/_phase7_backtest.py — load for display
summary_df = pd.read_csv('../outputs/predictions/backtest_summary.csv',
                         index_col=['model', 'horizon'])
summary_df

In [ ]:
from IPython.display import Image
Image('../figures/backtest_metrics_comparison.png')

**Key findings:**

**1. ARIMA beats Naive across all horizons and all error metrics.**
The improvement is modest but consistent — ~7% lower RMSE at h=1 (1.09 vs 1.18), maintained through h=21 (3.04 vs 3.25). For a near-unit-root series where the persistence baseline is hard to beat, a consistent edge across 36 folds is meaningful.

**2. Prophet fails badly on point forecast accuracy.**
Prophet's RMSE at h=1 is 5.64 — roughly 5× worse than ARIMA and 4.8× worse than Naive. Its trend component systematically overestimates VIX during the 2023–2026 low-volatility test window, pulling forecasts 3–5 pts above actuals. VIX's mean-reverting structure is the opposite of what Prophet's trend decomposition assumes.

**3. Directional accuracy tells a separate story.**
- **Naive** gets 0% by design — it always predicts flat (zero change), so it is always wrong on direction.
- **ARIMA** is best at h=1 (69.4%, well above the 50% random baseline). At h=21 it drops to 50.4% — essentially a coin flip. The mean-reversion signal decays with horizon.
- **Prophet** directional accuracy is reasonable (67% at h=1) but its magnitude errors are so large that correct direction with wrong magnitude is not useful.

**4. Spike onset is not captured by any model.**
The 2023–2026 test window was relatively calm. All three models lack an explicit mechanism for predicting abrupt volatility spikes — a fundamental limitation of linear models on VIX that would require regime-switching or GARCH-family extensions.

In [ ]:
Image('../figures/backtest_forecast_vs_actual.png')

## Final comparison table (mean across 36 folds)

| Model | h=1 RMSE | h=5 RMSE | h=21 RMSE | h=1 Dir Acc | h=21 Dir Acc | Verdict |
|-------|----------|----------|-----------|-------------|--------------|---------|
| Naive | 1.175 | 1.900 | 3.250 | 0% (flat) | 0% (flat) | Baseline floor |
| **ARIMA(1,0,2)** | **1.090** | **1.808** | **3.045** | **69.4%** | **50.4%** | **Best overall** |
| Prophet | 5.637 | 5.957 | 6.242 | 66.7% | 53.2% | Fails on magnitude |

**Winner: ARIMA(1,0,2)** beats the persistence baseline on every metric and every horizon. Prophet's additive trend decomposition is poorly suited to a mean-reverting index — it should not be used for VIX point forecasting without flat-trend mode or an explicit mean-reversion cap.

In [ ]:
prophet = ProphetForecaster(weekly_seasonality=True, yearly_seasonality=True)
prophet.fit(vix)
print('Prophet fitted.')

## Seasonality cross-check: day-of-week pattern in raw data

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']
dow = pd.DataFrame({'vix': vix, 'day': vix.index.day_name(), 'chg': vix.diff()})
dow_stats = dow.groupby('day')[['vix','chg']].mean().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(day_order, dow_stats['vix'], color='#2B6CB0', alpha=0.8)
axes[0].set_title('Mean VIX level by day of week')
axes[0].set_ylabel('Mean VIX')
axes[0].set_ylim(dow_stats['vix'].min() - 0.5, dow_stats['vix'].max() + 0.5)

colors = ['#E07070' if v > 0 else '#2B9348' for v in dow_stats['chg']]
axes[1].bar(day_order, dow_stats['chg'], color=colors, alpha=0.8)
axes[1].axhline(0, color='gray', linewidth=0.8)
axes[1].set_title('Mean daily VIX change by day of week')
axes[1].set_ylabel('Mean change (VIX pts)')

fig.tight_layout()
fig.savefig('../figures/vix_day_of_week.png', dpi=150)
plt.show()

spread = dow_stats['vix'].max() - dow_stats['vix'].min()
print(f'Monday-Friday mean spread : {spread:.3f} VIX pts  ({spread/vix.std()*100:.1f}% of std dev)')
print(dow_stats.round(4))

**Findings:**

There IS a day-of-week pattern in VIX: Monday is systematically highest (+0.357 avg change), Friday lowest (-0.174). This is the well-known *weekend volatility premium* — options price in the uncertainty of the two non-trading days.

However, the Monday–Friday **mean spread is only 0.44 VIX pts** (~6% of the series std dev of 7.75). This is real but small.

In [ ]:
amps = prophet.seasonality_amplitudes()
print('Prophet seasonality amplitudes (peak-to-trough):')
for name, amp in amps.items():
    print(f'  {name:10s}: {amp:.3f} VIX pts  ({amp/vix.std()*100:.1f}% of std dev)')

**Reconciliation — Phase 5 vs Phase 6:**

| Method | Weekly signal | Interpretation |
|--------|--------------|----------------|
| Raw data (day-of-week means) | **0.44 VIX pts** (6% of std) | Small but real weekend-volatility premium |
| ARIMA SMA(5) on full series | **0.0005, p=0.923** | Lag-5 autocorrelation adds no predictive power beyond AR |
| Prophet weekly amplitude | **~4 VIX pts** (~52% of std) | Overfitted Fourier terms — ~9× the actual signal |

These results are not contradictory:
- The day-of-week effect is **genuine but small** (confirmed by both raw data and ARIMA)
- ARIMA's SMA(5) tests whether the *lag-5 autocorrelation in residuals* adds information — it doesn't, because the AR(1)≈0.984 already absorbs most of the signal
- Prophet's weekly Fourier terms overfit to the small signal, amplifying it ~9× — this is a known risk when Fourier order is unconstrained relative to the signal strength

**Conclusion: the Phase 5 negative result stands.** Weekly seasonality is not a meaningful driver of VIX forecasting performance. Whether Prophet's overfit weekly component helps or hurts out-of-sample is an empirical question that Phase 7 will answer.

## Prophet component plot and in-sample fit

In [ ]:
from IPython.display import Image
Image('../figures/prophet_insample_fit.png')

## Prophet multi-step forecasts

In [ ]:
print(f'Current VIX ({vix.index[-1].date()}): {vix.iloc[-1]:.2f}')
print()
print(f'{"Horizon":>10}  {"ARIMA(1,0,2)":>14}  {"Prophet":>10}  {"Naive":>8}')
print('-' * 50)
from src.vix_forecasting.models.baseline import NaiveForecaster
naive = NaiveForecaster()
naive.fit(vix)
for h in (1, 5, 21):
    a = arima.predict(h)[-1]
    pr = prophet.predict(h)[-1]
    na = naive.predict(h)[-1]
    print(f'{h:>7d}d  {a:>14.3f}  {pr:>10.3f}  {na:>8.3f}')

**Observations:**

- At **horizon=1**, all three models are close — the AR(1)≈0.984 means the next day's VIX is almost always near today's.
- At **horizon=21**, models diverge: ARIMA mean-reverts upward toward ~19.3 (its implied long-run mean), Naive stays flat, and Prophet may diverge depending on trend and seasonal components. The relative out-of-sample performance across these regimes is what Phase 7 measures.

---

## Phase 5–6 Summary

| Property | Result |
|----------|--------|
| Integration order d | 0 — levels are stationary (Phase 3) |
| ARIMA order | (1, 0, 2) — best converged model by AIC |
| Weekly seasonality (ARIMA SMA5) | Rejected — coeff 0.0005, p=0.923 on full series |
| Day-of-week pattern (raw data) | Real but tiny — 0.44 VIX pts spread (6% of std) |
| Prophet weekly amplitude | ~4 VIX pts — Fourier overfitting of a small signal |
| Prophet yearly amplitude | ~3.3 VIX pts — plausible annual patterns (earnings, Fed) |
| All models: interface | fit() / predict(1,5,21) returning np.ndarray — verified |

Phase 7 (walk-forward backtest) will compare all three models quantitatively.